# 4.1 Introduction to Genomic Data

**Omics series, class 1 of 7.** *All of Us* has sequenced the genomes of more than 535,000 participants. In this class we learn what that data is, where it lives, how to connect it to health records, and how to pull out a small piece of it, for our **type 2 diabetes** cohort, without spending much money.

### By the end of this notebook you'll be able to

1. Explain DNA, variants, alleles, and genotypes in plain language
2. Describe the *All of Us* genomic data (short-read and long-read genomes, arrays) and its file formats
3. Find the genomic files in your workspace, and the ready-made extras (genetic ancestry, relatedness)
4. Link genomic data to the type 2 diabetes cohort, and choose a fair sample of cases and controls
5. Use **PLINK 2** to extract one region of the genome, and look at one well-known diabetes variant

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | the Controlled Tier dataset (BigQuery) and the v9 genomic files |
| 💾 **Saves** | `gwas_samples.csv` and the *TCF7L2* region files, used again in **4.2** |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 20 |
| 2. A tour of the genomic data | 15 |
| 3. The cohort, and who has genome data | 25 |
| 4. Genetic ancestry | 15 |
| 5. Extract one region with PLINK | 25 |
| 6. One famous variant | 15 |
| Wrap-up | 5 |

**Before you start:**

- Run **0.0 setup**, so `aou_setup.R` has your bucket and dataset
- This notebook is in `notebooks/omics/`. It loads `../aou_setup.R` and `../sql/t2d_cohort.sql` from the folder above, so keep the folders as they are in the repository

> 💰 **Cost:** runs on a **standard app** (no cluster). The cohort query costs a few cents to under a dollar; reading one small region of the genome takes seconds.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including genotypes and IDs), or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### DNA, genes, and the genome

- **DNA** is a long chain of four "letters" (bases): **A**, **C**, **G**, **T**.
- A person's **genome** is all of their DNA: about 3 billion letters, split into **23 pairs of chromosomes**. You get one of each pair from each parent.
- A **gene** is a stretch of DNA with the instructions for making a product, such as a protein.
- A **position** in the genome is written as **chromosome:position**, e.g. `chr10:112998590`, using a standard map of the genome called a **reference genome**. *All of Us* uses **GRCh38** (also called hg38). Positions on an older reference (GRCh37) are different, so always check which one a paper uses.

### Variants, alleles, and genotypes

People's DNA is 99.9% the same. The places where it differs are **variants**.

| Term | Means | Example |
|---|---|---|
| **SNP** (single-nucleotide polymorphism, say "snip") | One letter differs | some people have **C**, others **T** at `chr10:112998590` |
| **Indel** | A few letters inserted or deleted | |
| **Allele** | One version of a variant. The **reference** allele (REF) is the one in the reference genome; the **alternate** (ALT) is the other | REF = C, ALT = T |
| **Genotype** | The two alleles a person has (one from each parent), often counted as **0, 1, or 2** copies of the ALT allele | C/C = 0, C/T = 1, T/T = 2 |
| **Allele frequency** | How common an allele is in a group | ALT allele in 30% of chromosomes |
| **rsID** | A standard name for a known variant | **rs7903146** |

**Common** variants (allele frequency above about 1%) are shared by many people and are what most genome-wide studies test. **Rare** variants are found in few people.

### How *All of Us* measured DNA

| Data | What it is | Participants (v9) |
|---|---|---|
| **Short-read whole genome sequencing (srWGS)** | Reads the whole genome in short pieces: billions of variants | 535,662 |
| **Long-read whole genome sequencing (lrWGS)** | Long pieces: better for large and complex changes | 14,521 |
| **Genotyping arrays** | A chip that checks about 1.9 million chosen positions | 553,949 |

We use the **short-read genomes**. To make them easier to use, *All of Us* also provides smaller subsets:

| Subset | Contains |
|---|---|
| **ACAF** (*allele count / allele frequency* threshold) | Common variants: those above about 1% frequency, or carried by more than 100 people, in at least one genetic ancestry group. **This is what we use.** |
| **Exome** | Variants in the protein-coding parts of genes |
| **ClinVar** | Variants listed in ClinVar, a database of clinically reported variants |

### Today's example: *TCF7L2* and type 2 diabetes

The gene ***TCF7L2*** on chromosome 10 holds the strongest known genetic risk factor for type 2 diabetes. Each copy of the **T** allele of the variant **rs7903146** raises the risk by roughly 1.3 to 1.4 times in many populations. We'll find this variant in *All of Us* in Part 6, and run a full genome-wide-style analysis of the region in **4.2**.

## Part 2. A tour of the genomic data

### Setup

`source("../aou_setup.R")` loads the settings and helper functions from the folder above (`..` means "one folder up"). Two helpers are new for the omics notebooks:

| Helper | What it does |
|---|---|
| `aou_data_dir()` | finds the folder where the *All of Us* data files are mounted (its name differs between workspaces) |
| `hide_small_counts()` | blanks out counts of 1 to 20 in a table, as *All of Us* policy requires before sharing |

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

AOU_DIR <- aou_data_dir("v9")
AOU_DIR

list.files(AOU_DIR)

**🔍 Read the output:** the v9 data folder contains `wgs` (genomes), `microarray` (arrays), `multiomics` (RNA-seq and proteomics, classes 4.5–4.7), `known_issues`, and more.

Where every file lives is listed in the **CDR v9 data dictionary**, tab **Genomic and Multi-omics File Paths**.

### The short-read genome data

In [ ]:
SNPINDEL_DIR <- file.path(AOU_DIR, "wgs", "short_read", "snpindel")

list.files(SNPINDEL_DIR)                                # the subsets
list.files(file.path(SNPINDEL_DIR, "acaf_threshold"))   # the same data in different file formats

**🔍 Read the output:** the **ACAF** subset comes in several **file formats**. They hold the same genotypes, for different tools:

| Format | Files | Used by |
|---|---|---|
| **PLINK 2** (`pgen`) | `.pgen` (genotypes, compressed), `.pvar` (variants), `.psam` (samples) | PLINK 2: **what we use** |
| **PLINK 1** (`plink_bed`, `bed`) | `.bed`, `.bim`, `.fam` | PLINK 1.9 and many older tools |
| **BGEN** | `.bgen` | REGENIE, BOLT-LMM |
| **VCF** | `.vcf.gz` | the standard text format; most tools read it |
| **Hail** (`splitMT`, `multiMT`) | folders | Hail (Python): used in **4.3** |

Every format has **one file per chromosome**.

### How big is one chromosome?

`file.size()` reads only the file's size, not its contents, so this is instant and free.

In [ ]:
pgen_dir <- file.path(SNPINDEL_DIR, "acaf_threshold", "pgen")
bed_dir  <- file.path(SNPINDEL_DIR, "acaf_threshold", "plink_bed")

tibble(
  format = c("PLINK 2 (.pgen)", "PLINK 1 (.bed)"),
  chr10_size_GB = round(c(file.size(file.path(pgen_dir, "acaf_threshold.chr10.pgen")),
                          file.size(file.path(bed_dir,  "acaf_threshold.chr10.bed"))) / 1e9)
)

**🔍 Read the output:** one chromosome is tens to hundreds of **gigabytes**. The PLINK 2 file is about 10 times smaller because it's compressed. Reading a whole chromosome would take a long time and cost money, so in Part 5 we'll read **only one small region**: PLINK can jump straight to it.

### Ready-made extras: the `aux` folder

*All of Us* has already computed several things from the genomes, so you don't have to:

In [ ]:
AUX_DIR <- file.path(SNPINDEL_DIR, "aux")
list.files(AUX_DIR)

| Folder | Contains | We use it |
|---|---|---|
| `ancestry` | Each person's **genetic ancestry group** and **principal components (PCs)** | Part 4, and 4.2 |
| `relatedness` | Pairs of related people, and a list of samples to remove to keep only unrelated people | Part 3 |
| `pca` | PCs for everyone and for each ancestry group | |
| `admixture_estimates` | Estimated proportions of each ancestry per person | |
| `vat` | **Variant annotation table**: what each variant does (gene, effect, frequencies) | |
| `hla_variants`, `pgx`, `cmrg`, `mitochondria`, `phasing` | Immune genes (HLA), pharmacogenomics, hard-to-sequence genes, mitochondrial DNA, phasing | |

## Part 3. The cohort, and who has genome data

### The type 2 diabetes cohort

Every class in this series starts with the **same cohort**, from the shared query in `sql/t2d_cohort.sql`:

- **Cases:** age 18+, a type 2 diabetes diagnosis (concept 201826 or anything below it), and no type 1 diabetes diagnosis
- **Controls:** age 18+, no diabetes diagnosis of any type (concept 201820)
- Everyone has **EHR data** (at least one visit from a health care site, or records they shared themselves)

`read_file()` reads the SQL; `str_glue()` fills in the dataset and the date age is calculated on (the v9 data cutoff, 2025-01-01). Open `sql/t2d_cohort.sql` to read the query: every step has a comment.

💰 This query costs a few cents to under a dollar, and takes a minute or two.

In [ ]:
cohort_sql <- read_file("../sql/t2d_cohort.sql")

cohort <- run_query(str_glue(cohort_sql, AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

cohort %>% count(cohort)

**🔍 Read the output:** *case*, *control*, and *excluded* (for example people with both type 1 and type 2 codes). We keep only cases and controls.

`person_id` is turned into **text** with `as.character()`, because the genome files store IDs as text. A number and the same digits as text **don't match** in a join (see **2.3 join_and_bind**).

### Who has a genome?

The `.psam` file lists every person in the genome files: one row per person, with their ID in the column `#IID`. We read only that list, not the genotypes.

In [ ]:
psam <- read_tsv(file.path(pgen_dir, "acaf_threshold.chr10.psam"),
                 col_types = cols(.default = "c"))
names(psam)
nrow(psam)   # people with short-read genomes

genome_ids <- psam[["#IID"]]

cohort <- cohort %>% mutate(has_genome = person_id %in% genome_ids)
cohort %>% filter(cohort != "excluded") %>% count(cohort, has_genome)

**🔍 Read the output:**

- `#IID` is the participant's ID: the **same number as `person_id`** in the health data. That's what links genomes to health records.
- The `SEX` column in this file is empty, so we take **sex at birth** from the cohort query instead.
- Most people in our cohort have a genome, but not all.

### Remove related people

Relatives share long stretches of DNA. Most genetic analyses assume people are **unrelated**, so related pairs make results look more certain than they are. *All of Us* provides a list of samples to remove so that no two remaining people are closely related.

In [ ]:
related <- read_tsv(file.path(AUX_DIR, "relatedness", "relatedness_flagged_samples.tsv"),
                    col_types = cols(.default = "c"))
names(related)
nrow(related)   # samples to remove

mean(related$sample_id %in% genome_ids)   # share of these IDs found in the genome file: should be 1

### Choose the samples for this series

We keep cases and controls who have a genome, aren't on the relatedness list, are **18 to 100** years old (a few recorded ages above 100 are probably data errors; see **2.1 clean**), and have sex at birth recorded as *Female* or *Male* (needed as a covariate).

Then we take a **random sample of 2,500 cases and 2,500 controls**. That's enough to see the *TCF7L2* signal clearly, and keeps every step fast and cheap. `set.seed()` makes the sample the same every time, for everyone.

| Function | What it does |
|---|---|
| `%in%` | is each value in this list? (`!` means "not") |
| `between(age, 18, 100)` | is age from 18 to 100? |
| `slice_sample(n = …)` | picks that many rows at random, **within each group** of `group_by()` |

In [ ]:
N_PER_GROUP <- 2500

eligible <- cohort %>%
  filter(cohort %in% c("case", "control"),
         has_genome,
         !person_id %in% related$sample_id,
         between(age, 18, 100),
         sex_at_birth %in% c("Female", "Male"))

eligible %>% count(cohort)

set.seed(2026)
gwas_samples <- eligible %>%
  group_by(cohort) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

gwas_samples %>% count(cohort)

**🔍 Read the output:** the first table shows how many people are **eligible**; the second, the **2,500 + 2,500** we sampled.

**💬 Discuss:** the sample is random *within* each group, but cases and controls differ in other ways: age, for example. How could that affect a comparison of their genes? (We adjust for age, sex, and ancestry in 4.2.)

### ✍️ Your turn

How many eligible people were removed because they were on the relatedness list? (Hint: count cases and controls with a genome **before** and **after** that filter.)

<details>
<summary><b>Show answer</b></summary>

```r
cohort %>%
  filter(cohort %in% c("case", "control"), has_genome) %>%
  count(related = person_id %in% related$sample_id)
```

`TRUE` is the number removed. If any count is between 1 and 20, don't share it.

</details>

In [ ]:
# Your code here

## Part 4. Genetic ancestry

### Why ancestry matters

Allele frequencies differ between populations, and so do the rates of many diseases, for genetic and non-genetic reasons. If we compare cases and controls without accounting for ancestry, a variant can look linked to diabetes just because it's more common in a group that also has more diabetes. This is called **confounding by ancestry** (or *population stratification*).

The usual fix is to adjust for **principal components (PCs)** of the genotypes: a few numbers per person that summarise their genetic ancestry. People with similar ancestry have similar PCs.

> **Genetic ancestry** is estimated from DNA. It's **not the same** as self-reported race or ethnicity, which is a social category.

`ancestry_preds.tsv` has, for each person with a genome:

| Column | Contains |
|---|---|
| `research_id` | the participant ID (same as `person_id`) |
| `ancestry_pred` | the predicted ancestry group: `afr`, `amr`, `eas`, `eur`, `mid`, `sas`, or `oth` (none clearly) |
| `pca_features` | the PCs, stored as one text value like `[0.12, -0.03, …]` |

In [ ]:
ancestry <- read_tsv(file.path(AUX_DIR, "ancestry", "ancestry_preds.tsv"),
                     col_types = cols(.default = "c"))
names(ancestry)

The PCs are stored as one piece of text per person, so we split them into separate number columns, `PC1`, `PC2`, …:

| Step | Function |
|---|---|
| remove the `[` and `]` | `str_remove_all(pca_features, "\\[|\\]")` |
| split at the commas into columns | `separate_wider_delim()` |
| turn the text into numbers | `mutate(across(starts_with("PC"), as.numeric))` |

Then we **join** the PCs to our 5,000 people with `left_join()`, matching `person_id` to `research_id`.

In [ ]:
n_pcs <- str_count(ancestry$pca_features[1], ",") + 1   # how many PCs per person
n_pcs

pcs <- ancestry %>%
  transmute(person_id = research_id,
            ancestry  = ancestry_pred,
            pca_features = str_remove_all(pca_features, "\\[|\\]")) %>%
  separate_wider_delim(pca_features, delim = ",", names = paste0("PC", 1:n_pcs)) %>%
  mutate(across(starts_with("PC"), as.numeric))

gwas_samples <- gwas_samples %>% left_join(pcs, by = "person_id")

gwas_samples %>% count(cohort, ancestry) %>% hide_small_counts()

**🔍 Read the output:** our sample includes several ancestry groups. `NA` in the `n` column means a count of 1 to 20, hidden by `hide_small_counts()`.

### Plot the first two PCs

In [ ]:
ggplot(gwas_samples, aes(PC1, PC2, colour = ancestry)) +
  geom_point(alpha = 0.5, size = 1) +
  labs(title = "Genetic principal components", colour = "Genetic ancestry")

**🔍 Read the output:** each dot is a person. People in the same ancestry group cluster together; people with mixed ancestry fall in between. *All of Us* is very diverse, so many people are between the clusters. This is why we'll adjust for PCs in 4.2, rather than analysing one group at a time.

👩‍🏫 **Instructor tip:** a newer approach, **genomic-relatedness-matched association (GRMA)**, compares people with their closest relatives instead of adjusting for PCs (All of Us Office Hours, 2026-08-28). A good discussion point on why PCs may not remove all confounding.

## Part 5. Extract one region with PLINK

**PLINK 2** is a free command-line program for genetic data. It's already installed in the app. R can run it with `system()`, which runs a command as if typed in the Terminal.

We'll ask PLINK for:

| Option | Means |
|---|---|
| `--pfile …/acaf_threshold.chr10` | read the PLINK 2 files for chromosome 10 (`.pgen`, `.pvar`, `.psam`) |
| `--keep keep.txt` | keep only the people listed in this file: our 5,000 |
| `--chr 10 --from-bp … --to-bp …` | keep only variants in this region (positions in base pairs, GRCh38) |
| `--make-pgen` | save the result as new, small PLINK 2 files |
| `--freq` | also calculate each variant's allele frequency |
| `--out …` | the start of the output file names |

The region is ***TCF7L2*** and the DNA around it: chromosome 10, from 112.7 to 113.4 million.

We save the files in our **bucket** (`DATA_DIR`), so 4.2 can use them.

In [ ]:
GENO_DIR <- file.path(DATA_DIR, "genomics")
dir.create(GENO_DIR, showWarnings = FALSE)

# 1. The list of people to keep: one ID per line
keep_file <- file.path(GENO_DIR, "keep.txt")
write_lines(gwas_samples$person_id, keep_file)

# 2. The PLINK command
REGION_START <- 112700000
REGION_END   <- 113400000
out_prefix   <- file.path(GENO_DIR, "tcf7l2")

plink_cmd <- str_glue(
  "plink2 --pfile {shQuote(file.path(pgen_dir, 'acaf_threshold.chr10'))} ",
  "--keep {shQuote(keep_file)} ",
  "--chr 10 --from-bp {REGION_START} --to-bp {REGION_END} ",
  "--make-pgen --freq ",
  "--out {shQuote(out_prefix)}"
)
cat(plink_cmd)

**🔍 Read the output:** this is the exact command R will run. `shQuote()` puts quotes around file paths, in case they contain spaces.

Now run it. It takes about 10 to 30 seconds: PLINK jumps to the region instead of reading the whole 80 GB file.

`system(…, intern = TRUE)` runs the command and **keeps** what PLINK prints, so we can show it here: in Jupyter, a plain `system()` call prints nowhere you can see. What PLINK prints includes progress counters (`0%… 1%…`) that look messy in a notebook, so we show the key lines from the **log file** PLINK saves (`tcf7l2.log`) instead.

In [ ]:
plink_output <- system(plink_cmd, intern = TRUE)

# The key lines from PLINK's log file: how many samples and variants it kept
plink_log <- read_lines(paste0(out_prefix, ".log"))
str_subset(plink_log, "samples|variants")

list.files(GENO_DIR)

**🔍 Read the output:** PLINK's **log** of what it did. Look for:

- `535662 samples (0 females, 0 males, 535662 ambiguous…) loaded`: everyone in the genome file. Sex is "ambiguous" because the genome files don't include it; we take sex at birth from the survey data instead
- `--keep: … samples remaining`: should be our 5,000 (or slightly fewer if some IDs weren't found)
- `… variants remaining after main filters`: the variants in our region

The new files are small: `tcf7l2.pgen`, `.pvar`, `.psam`, the allele frequencies (`tcf7l2.afreq`), and the log (`tcf7l2.log`).

### Look at the variants

The `.pvar` file is a table of the variants: chromosome, position, ID, REF and ALT alleles. Lines starting with `##` are notes, so we skip them with `comment = "##"`.

In [ ]:
pvar <- read_tsv(paste0(out_prefix, ".pvar"), comment = "##", col_types = cols(.default = "c"))
names(pvar)
nrow(pvar)   # variants in the region
head(pvar)

# ALT_FREQS is text: a variant with two ALT alleles has two frequencies, e.g. "0.27,0".
# For the plot we keep the first ALT allele's frequency.
afreq <- read_tsv(paste0(out_prefix, ".afreq"), col_types = cols(.default = "c")) %>%
  mutate(alt1_freq = as.numeric(str_extract(ALT_FREQS, "^[^,]+")))

ggplot(afreq, aes(alt1_freq)) +
  geom_histogram(bins = 50) +
  labs(x = "ALT allele frequency in our 5,000 people", y = "Number of variants",
       title = "Allele frequencies in the TCF7L2 region")

**🔍 Read the output:**

- About **14,000 variants** in a 700,000-letter region: roughly one every 50 letters.
- The `ID` column is **not** an rsID like `rs7903146`. In *All of Us* it's **chromosome:position:REF**, e.g. `chr10:112998590:C`. So we find variants by **position**, as we do next.
- Some variants have **two ALT alleles**, e.g. `ALT` = `T,G`. These are **multiallelic** variants: at that position, people can have C, T, or G.
- `INFO` has notes about each variant across **all** *All of Us* genomes: `AF` is the ALT allele frequency in everyone (not just our sample), `AC` the number of copies, `AN` the number of alleles checked.
- In the plot, most variants have a **low** ALT frequency: even in this "common variant" subset, most are rare in any one sample.


## Part 6. One famous variant: rs7903146

rs7903146 is at **chr10:112998590** (GRCh38). We find it by position in the `.pvar` table:

In [ ]:
LEAD_POS <- 112998590

lead_variant <- pvar %>% filter(POS == LEAD_POS)
lead_variant

afreq %>% filter(ID %in% lead_variant$ID)

**🔍 Read the output:** rs7903146 is **multiallelic** here: `REF` is **C**, `ALT` is **T,G**. T is common (`AF=0.276` in `INFO`: about 28% of all *All of Us* alleles); G is extremely rare. **T** is the allele linked to higher diabetes risk. In `afreq`, `ALT_FREQS` shows two values, one for T and one for G.

### Each person's genotype

`--export A` writes each person's genotype as **0, 1, or 2 copies** of one allele (the column name ends with the allele counted, e.g. `_T`). We export just this one position.

For a multiallelic variant, PLINK counts the **REF** allele by default (here C, the *non*-risk allele). To count **T**, we give PLINK a small file with the variant ID and the allele to count (`--export-allele`).


In [ ]:
lead_prefix <- file.path(GENO_DIR, "rs7903146")

# Which allele to count: one line with the variant ID and the allele
allele_file <- file.path(GENO_DIR, "count_allele.txt")
write_lines(str_glue("{lead_variant$ID[1]}\tT"), allele_file)

export_output <- system(str_glue(
  "plink2 --pfile {shQuote(out_prefix)} --chr 10 --from-bp {LEAD_POS} --to-bp {LEAD_POS} ",
  "--export A --export-allele {shQuote(allele_file)} --out {shQuote(lead_prefix)}"
), intern = TRUE)

lead <- read_tsv(paste0(lead_prefix, ".raw"), col_types = cols(.default = "c"))
names(lead)

**🔍 Read the output:** the first columns describe the person (`FID`, `IID`, …); the **last column** is the genotype. Its name shows the variant and the allele counted: it should end in `_T`.

Now **join** the genotypes to the cohort and compare cases with controls. `IID` is the person's ID.

In [ ]:
geno_col <- names(lead)[ncol(lead)]   # the genotype column (the last one)
geno_col

lead_data <- gwas_samples %>%
  left_join(lead %>% transmute(person_id = IID, dosage = as.numeric(.data[[geno_col]])),
            by = "person_id")

# Genotype counts (0, 1, or 2 copies) in cases and controls
lead_data %>% count(cohort, dosage) %>% hide_small_counts()

# Frequency of the counted allele in each group: copies / (2 x people)
lead_data %>%
  group_by(cohort) %>%
  summarise(allele_freq = round(sum(dosage, na.rm = TRUE) / (2 * sum(!is.na(dosage))), 3))

**🔍 Read the output:** the T allele should be **more common in cases** than in controls (about 30% vs 27% in our test run). That's the *TCF7L2* signal. One person has no genotype (`NA`) at this position; with 1 to 20 people, the count is hidden.

### A first odds ratio

A logistic regression of case status on the number of copies gives the **odds ratio per allele**: how much the odds of type 2 diabetes change with each extra copy. We'll add age, sex, and ancestry PCs in 4.2; this one is **unadjusted**.

In [ ]:
lead_data <- lead_data %>% mutate(case = if_else(cohort == "case", 1, 0))

glm(case ~ dosage, data = lead_data, family = binomial) %>%
  broom::tidy(exponentiate = TRUE, conf.int = TRUE) %>%
  filter(term == "dosage")

**🔍 Read the output:** `estimate` is the **odds ratio per copy** of the T allele: above 1 means more copies, higher odds of type 2 diabetes. `conf.low` and `conf.high` are the 95% confidence interval; if it doesn't include 1, the association is unlikely to be chance.

Published studies report about **1.3 to 1.4** per T allele, mostly in people of European ancestry. Ours (about 1.2 in our test run) is **smaller**. Why might that be?

- **Mixed ancestry, no adjustment:** the T allele's frequency differs between ancestry groups, and so does our case/control mix
- **Case definition:** cases come from diagnosis codes in health records; some may be misclassified
- **Controls:** some younger controls will develop diabetes later
- **Sample size:** 5,000 people give a wide confidence interval

**💬 Discuss:** this used **one** variant. In 4.2 we test **all 14,000** variants in the region, adjust for age, sex, and ancestry, and see whether rs7903146 comes out on top.

### ✍️ Your turn

Is the T allele equally common in every ancestry group? Calculate the allele frequency of T by `ancestry` (controls only).

<details>
<summary><b>Show answer</b></summary>

```r
lead_data %>%
  filter(cohort == "control") %>%
  group_by(ancestry) %>%
  summarise(people = n(),
            allele_freq = round(sum(dosage, na.rm = TRUE) / (2 * sum(!is.na(dosage))), 3))
```

Frequencies differ between groups. This is exactly why we adjust for ancestry: without it, a difference between cases and controls could partly reflect a difference in their ancestry mix. Don't share rows where `people` is 1 to 20.

</details>


In [ ]:
# Your code here

## Save for 4.2

`gwas_samples` has everything 4.2 needs about each person: cohort, age, sex at birth, ancestry, and PCs. It contains participant-level data, so it stays in your **bucket**, inside the Workbench.

In [ ]:
save_data(gwas_samples, "gwas_samples.csv")

## ⚠️ Common errors

| Message | Means |
|---|---|
| *cannot open file '../aou_setup.R'* | The notebook isn't in `notebooks/omics/`. Move it there, or `setwd()` to that folder |
| *Can't find the All of Us v9 data folder* | The Controlled Tier data collection isn't added to this workspace |
| *plink2: command not found* | Use an *All of Us* app: PLINK is installed there |
| *--keep: 0 samples remaining* | The IDs in `keep.txt` don't match the genome IDs. Check they're text, not numbers like `1e+06` |
| *Error in separate_wider_delim()* | Some rows have a different number of PCs; add `too_few = "align_start"` |

## ✅ Key takeaways

- A **variant** is a place where people's DNA differs; a **genotype** counts copies of an allele (0, 1, 2)
- *All of Us* genomes are huge: **read only the region you need** (PLINK can jump to it)
- The **participant ID** links genomes to health records; IDs must be the **same type** (text) to match
- Use the ready-made **relatedness** and **ancestry** files instead of computing your own
- **Genetic ancestry** is a confounder in genetic studies; adjust for it with **PCs**

**Next:** **4.2 gwas_plink**: test every variant in the *TCF7L2* region.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).